In [1]:
import torch
from src.xl_wrapper import RuGPT3XL
from torch.nn.modules.utils import consume_prefix_in_state_dict_if_present
import json


[2024-09-29 18:17:35,996] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


df: /root/.triton/autotune: No such file or directory


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
model_path = 'xl_old/poetry'

In [3]:
#| export
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("./tokenizer/rugpt3xl.tokenizer", local_files_only=True)


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
#| export
seq_length = 512

from src.xl_wrapper import RuGPT3XL
model = RuGPT3XL.from_pretrained(
    "sberbank-ai/rugpt3xl",
    weights_path=f"./models/{model_path}.model",
    deepspeed_config_path="src/deepspeed_config/gpt3_xl_2048.json",
    seq_len=seq_length,
)
tokenizer = model.tokenizer

> initializing model parallel with size 1
[2024-09-29 18:17:38,893] [INFO] [config.py:733:__init__] Config mesh_device None world_size = 1


[W929 18:17:38.634513099 socket.cpp:752] [c10d] The client socket cannot be initialized to connect to [localhost]:6000 (errno: 97 - Address family not supported by protocol).
/workspaces/gpt/src/xl_wrapper.py:84: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. 

In [5]:
# Get the state dict
state_dict = model.state_dict()
consume_prefix_in_state_dict_if_present(state_dict, "module.")

# 1. Analyze tensor shapes
tensor_shapes = {k: v.shape for k, v in state_dict.items()}

# 2. Analyze components
component_details = {}
for name, module in model.named_modules():
    component_details[name] = {
        'type': module.__class__.__name__,
        'parameters': {param_name: param.shape for param_name, param in module.named_parameters()},
    }
    if hasattr(module, 'input_size'):
        component_details[name]['input_size'] = module.input_size
    if hasattr(module, 'output_size'):
        component_details[name]['output_size'] = module.output_size
    if hasattr(module, 'normalized_shape'):
        component_details[name]['normalized_shape'] = module.normalized_shape
    if hasattr(module, 'eps'):
        component_details[name]['eps'] = module.eps
    if hasattr(module, 'elementwise_affine'):
        component_details[name]['elementwise_affine'] = module.elementwise_affine

# Print results
print("1. Tensor shapes:")
print(json.dumps(tensor_shapes, indent=2))

print("\n2. Component details:")
print(json.dumps(component_details, indent=2))

# Additional analysis of attention mechanism
print("\n3. Attention mechanism details:")
for name, module in model.named_modules():
    if 'GPT3ParallelSelfAttention' in module.__class__.__name__:
        print(f"Attention module: {name}")
        print(f"  Query-Key-Value combined: {'query_key_value' in dict(module.named_children())}")
        if hasattr(module, 'num_attention_heads'):
            print(f"  Number of attention heads: {module.num_attention_heads}")
        if hasattr(module, 'hidden_size'):
            print(f"  Hidden size: {module.hidden_size}")

1. Tensor shapes:
{
  "model.word_embeddings.weight": [
    50264,
    2048
  ],
  "model.position_embeddings.weight": [
    2048,
    2048
  ],
  "model.transformer.layers.0.input_layernorm.weight": [
    2048
  ],
  "model.transformer.layers.0.input_layernorm.bias": [
    2048
  ],
  "model.transformer.layers.0.attention.query_key_value.weight": [
    6144,
    2048
  ],
  "model.transformer.layers.0.attention.query_key_value.bias": [
    6144
  ],
  "model.transformer.layers.0.attention.dense.weight": [
    2048,
    2048
  ],
  "model.transformer.layers.0.attention.dense.bias": [
    2048
  ],
  "model.transformer.layers.0.post_attention_layernorm.weight": [
    2048
  ],
  "model.transformer.layers.0.post_attention_layernorm.bias": [
    2048
  ],
  "model.transformer.layers.0.mlp.dense_h_to_4h.weight": [
    8192,
    2048
  ],
  "model.transformer.layers.0.mlp.dense_h_to_4h.bias": [
    8192
  ],
  "model.transformer.layers.0.mlp.dense_4h_to_h.weight": [
    2048,
    8192
  ],


/workspaces/gpt/src/fp16/fp16.py:75: FutureWarning: Positional args are being deprecated, use kwargs instead. Refer to https://pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.state_dict for details.
  return self.module.state_dict(destination, prefix, keep_vars)


In [6]:
import torch
from transformers import GPT2LMHeadModel, GPT2Config

def load_rugpt3xl_weights(model_path):
    return torch.load(model_path, map_location='cpu')

def create_gpt2_model(vocab_size=50264, n_positions=2048, n_embd=2048, n_layer=24, n_head=16):
    config = GPT2Config(
        vocab_size=vocab_size,
        n_positions=n_positions,
        n_embd=n_embd,
        n_layer=n_layer,
        n_head=n_head,
        n_inner=4*n_embd,
        activation_function='gelu',
        resid_pdrop=0.1,
        embd_pdrop=0.1,
        attn_pdrop=0.1,
        layer_norm_epsilon=1e-5
    )
    return GPT2LMHeadModel(config)

def convert_attention_weights(rugpt_weights, gpt2_weights, layer_index):
    # Convert the combined QKV weights to separate Q, K, V
    qkv_weight = rugpt_weights[f'module.module.transformer.layers.{layer_index}.attention.query_key_value.weight']
    qkv_bias = rugpt_weights[f'module.module.transformer.layers.{layer_index}.attention.query_key_value.bias']
    
    hidden_size = 2048
    num_heads = 16
    head_size = hidden_size // num_heads
    
    q_weight, k_weight, v_weight = qkv_weight.split(hidden_size, dim=0)
    q_bias, k_bias, v_bias = qkv_bias.split(hidden_size)
    
    gpt2_weights[f'transformer.h.{layer_index}.attn.c_attn.weight'] = torch.cat([q_weight, k_weight, v_weight], dim=0).t()
    gpt2_weights[f'transformer.h.{layer_index}.attn.c_attn.bias'] = torch.cat([q_bias, k_bias, v_bias])
    
    # Convert the output projection
    gpt2_weights[f'transformer.h.{layer_index}.attn.c_proj.weight'] = rugpt_weights[f'module.module.transformer.layers.{layer_index}.attention.dense.weight'].t()
    gpt2_weights[f'transformer.h.{layer_index}.attn.c_proj.bias'] = rugpt_weights[f'module.module.transformer.layers.{layer_index}.attention.dense.bias']

def convert_mlp_weights(rugpt_weights, gpt2_weights, layer_index):
    # Convert the MLP weights
    gpt2_weights[f'transformer.h.{layer_index}.mlp.c_fc.weight'] = rugpt_weights[f'module.module.transformer.layers.{layer_index}.mlp.dense_h_to_4h.weight'].t()
    gpt2_weights[f'transformer.h.{layer_index}.mlp.c_fc.bias'] = rugpt_weights[f'module.module.transformer.layers.{layer_index}.mlp.dense_h_to_4h.bias']
    gpt2_weights[f'transformer.h.{layer_index}.mlp.c_proj.weight'] = rugpt_weights[f'module.module.transformer.layers.{layer_index}.mlp.dense_4h_to_h.weight'].t()
    gpt2_weights[f'transformer.h.{layer_index}.mlp.c_proj.bias'] = rugpt_weights[f'module.module.transformer.layers.{layer_index}.mlp.dense_4h_to_h.bias']

def convert_layer_norm_weights(rugpt_weights, gpt2_weights, layer_index):
    # Convert the layer norm weights
    gpt2_weights[f'transformer.h.{layer_index}.ln_1.weight'] = rugpt_weights[f'module.module.transformer.layers.{layer_index}.input_layernorm.weight']
    gpt2_weights[f'transformer.h.{layer_index}.ln_1.bias'] = rugpt_weights[f'module.module.transformer.layers.{layer_index}.input_layernorm.bias']
    gpt2_weights[f'transformer.h.{layer_index}.ln_2.weight'] = rugpt_weights[f'module.module.transformer.layers.{layer_index}.post_attention_layernorm.weight']
    gpt2_weights[f'transformer.h.{layer_index}.ln_2.bias'] = rugpt_weights[f'module.module.transformer.layers.{layer_index}.post_attention_layernorm.bias']

def convert_rugpt3xl_to_gpt2(rugpt_model_path, output_path):
    # Load RuGPT3XL weights
    rugpt_weights = load_rugpt3xl_weights(rugpt_model_path)
    
    # Create a new GPT-2 model
    gpt2_model = create_gpt2_model()
    gpt2_weights = gpt2_model.state_dict()
    
    # Convert embedding weights
    gpt2_weights['transformer.wte.weight'] = rugpt_weights['module.module.word_embeddings.weight']
    gpt2_weights['transformer.wpe.weight'] = rugpt_weights['module.module.position_embeddings.weight']
    
    # Convert transformer layers
    for i in range(24):  # Assuming 24 layers
        convert_attention_weights(rugpt_weights, gpt2_weights, i)
        convert_mlp_weights(rugpt_weights, gpt2_weights, i)
        convert_layer_norm_weights(rugpt_weights, gpt2_weights, i)
    
    # Convert final layer norm
    gpt2_weights['transformer.ln_f.weight'] = rugpt_weights['module.module.transformer.final_layernorm.weight']
    gpt2_weights['transformer.ln_f.bias'] = rugpt_weights['module.module.transformer.final_layernorm.bias']
    
    # Set the converted weights
    gpt2_model.load_state_dict(gpt2_weights)
    
    # Save the converted model
    gpt2_model.save_pretrained(output_path)
    print(f"Converted model saved to {output_path}")

# Usage
rugpt_model_path = f"./models/{model_path}.model"
output_path = "./converted_gpt2_model"
convert_rugpt3xl_to_gpt2(rugpt_model_path, output_path)

/tmp/ipykernel_8239/2884282138.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(model_path, map_location='cpu')


Converted model saved to ./converted_gpt2_model


In [7]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("./tokenizer/rugpt3xl.tokenizer", local_files_only=True)    
tokenizer.save_pretrained(output_path)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


('./converted_gpt2_model/tokenizer_config.json',
 './converted_gpt2_model/special_tokens_map.json',
 './converted_gpt2_model/vocab.json',
 './converted_gpt2_model/merges.txt',
 './converted_gpt2_model/added_tokens.json')

In [8]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

def load_model_and_tokenizer(model_path):
    # Load the converted model
    model = GPT2LMHeadModel.from_pretrained(model_path)
    
    # Load the tokenizer
    # Note: Make sure to use the correct tokenizer for your model
    tokenizer = GPT2Tokenizer.from_pretrained(model_path)
    
    return model, tokenizer

def generate_text(model, tokenizer, prompt, max_length=100, num_return_sequences=1):
    # Encode the input prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    
    # Generate text
    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        no_repeat_ngram_size=2,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )
    
    # Decode and return the generated text
    return [tokenizer.decode(seq, skip_special_tokens=True) for seq in output]



In [9]:
# Path to the converted model
model_path = "./converted_gpt2_model"

# Load the model and tokenizer
model, tokenizer = load_model_and_tokenizer(model_path)

# Set the model to evaluation mode
model.eval()

# Test prompts
prompts = [
    "Однажды в студеную зимнюю пору",
    "Искусственный интеллект в современном мире",
    "Какие преимущества и недостатки у",
]

# Generate text for each prompt
for prompt in prompts:
    print(f"Prompt: {prompt}")
    generated_texts = generate_text(model, tokenizer, prompt)
    for i, text in enumerate(generated_texts):
        print(f"Generated text {i+1}:")
        print(text)
        print()
    print("-" * 50)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpe

Prompt: Однажды в студеную зимнюю пору


/usr/local/lib/python3.10/dist-packages/transformers/modeling_attn_mask_utils.py:369: FutureWarning: `torch._dynamo.external_utils.is_compiling` is deprecated. Use `torch.compiler.is_compiling` instead.
  or (hasattr(torch, "_dynamo") and torch._dynamo.is_compiling())
/usr/local/lib/python3.10/dist-packages/transformers/modeling_attn_mask_utils.py:259: FutureWarning: `torch._dynamo.external_utils.is_compiling` is deprecated. Use `torch.compiler.is_compiling` instead.
  or (hasattr(torch, "_dynamo") and torch._dynamo.is_compiling())
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated text 1:
Однажды в студеную зимнюю пору
Я из лесу вышел; был сильный мороз.
Гляжу, поднимается медленно в гору
Лошадка, везущая хворосту воз. 

В.А.Жуковскому
Не знаю, как и почему,
Но я люблю тебя, Россия, -
И в этом нет ни капли лести, ни
Малейшей доли самовнушенья. 
Люблю твой снег, твою пургу,  Твой ветер

--------------------------------------------------
Prompt: Искусственный интеллект в современном мире


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated text 1:
Искусственный интеллект в современном мире

В. А. Шемшук
Искусство и наука
(О книге «Искусственные интеллекты в современной жизни»)
«Искусственным интеллектом» называют систему, которая может решать задачи, которые не может решить человек.
Это не значит, что она может делать то, чего не могут делать люди. Это значит только,
что она решает задачи не хуже, а лучше людей. И это не просто слова. В
настоящее время

--------------------------------------------------
Prompt: Какие преимущества и недостатки у
Generated text 1:
Какие преимущества и недостатки у этого вида спорта?
Плюсы: - Не требует больших затрат на оборудование и инвентарь. - Можно заниматься в любом возрасте. Минусы: Не подходит для людей с избыточным весом.
Какие плюсы и минусы у этой игры? (см. внутри)
Минусы - это то, что она не для всех. А плюсы - что это игра.
А для кого она? )))))
Для всех, кто любит играть в игры. И для

--------------------------------------------------
